In [1]:
import duckdb

con = duckdb.connect()
con.sql("""
    CREATE VIEW ist AS
    SELECT * FROM read_csv_auto('../data/raw/IST_corrected.csv')
""")
con.sql("DESCRIBE ist").df()

InvalidInputException: Invalid Input Error: CSV Error on Line: 147
Original Line: 
5,46,F,M,52,N,,Y,Y,,,140,N,N,N,N,N,N,Y,N,POCS,wrz-91,99,99,2,N,N,N,Y,N,,N,3,,,N,N,N,N,N,N,,,N,,,N,,,Y,N,N,N,,N,,N,,,,N,,Y,3,,N,,0,,N,,,,,N,N,A,Y,N,29,185,PORT,19,,,Y,Y,0,185,0.1527,0.0223,0.0117,1,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Invalid unicode (byte sequence mismatch) detected. This file is not utf-8 encoded.

Possible Solution: Set the correct encoding, if available, to read this CSV File (e.g., encoding='UTF-16')
Possible Solution: Enable ignore errors (ignore_errors=true) to skip this row

  file = ../data/raw/IST_corrected.csv
  delimiter = , (Auto-Detected)
  quote = " (Auto-Detected)
  escape = " (Auto-Detected)
  new_line = Single-Line File (Auto-Detected)
  header = false (Auto-Detected)
  skip_rows = 0 (Auto-Detected)
  comment = (empty) (Auto-Detected)
  strict_mode = true (Auto-Detected)
  date_format =  (Auto-Detected)
  timestamp_format =  (Auto-Detected)
  null_padding = 0
  sample_s

In [2]:
import re
from pathlib import Path

raw = Path("../data/raw/IST_corrected.csv").read_bytes()

print("dimensione:", len(raw), "byte")
print("inizio file:", raw[:200])
print(
    "fine riga \\r\\n:",
    raw.count(b"\r\n"),
    " \\n:",
    raw.count(b"\n"),
    " \\r:",
    raw.count(b"\r"),
)

nonascii = sorted({b for b in raw if b > 127})
print("byte non-ASCII presenti:", [hex(b) for b in nonascii])

for m in list(re.finditer(rb"[\x80-\xff]+", raw))[:5]:
    print(raw[max(0, m.start() - 15) : m.end() + 15])

dimensione: 4798288 byte
inizio file: b'HOSPNUM,RDELAY,RCONSC,SEX,AGE,RSLEEP,RATRIAL,RCT,RVISINF,RHEP24,RASP3,RSBP,RDEF1,RDEF2,RDEF3,RDEF4,RDEF5,RDEF6,RDEF7,RDEF8,STYPE,RDATE,HOURLOCAL,MINLOCAL,DAYLOCAL,RXASP,RXHEP,DASP14,DASPLT,DLH14,DMH14'
fine riga \r\n: 19436  \n: 19436  \r: 19436
byte non-ASCII presenti: ['0x9f']
b'Y,Y,N,N,TACS,pa\x9f-91,99,99,3,Y,L'
b'N,N,N,N,PACS,pa\x9f-91,99,99,4,Y,L'
b'N,N,N,N,PACS,pa\x9f-91,99,99,4,N,N'
b'C,C,N,N,PACS,pa\x9f-91,99,99,5,N,H'
b'N,N,N,N,PACS,pa\x9f-91,99,99,5,Y,N'


In [3]:
text = raw.decode("cp1250")
assert "paź" in text, "decodifica inattesa: controllare l'encoding"

out = Path("../data/interim/IST_corrected_utf8.csv")
out.write_bytes(text.encode("utf-8"))
print("scritto:", out, out.stat().st_size, "byte")

scritto: ..\data\interim\IST_corrected_utf8.csv 4799920 byte


In [4]:
con.sql("""
    CREATE OR REPLACE VIEW ist AS
    SELECT * FROM read_csv_auto('../data/interim/IST_corrected_utf8.csv')
""")
print(con.sql("SELECT COUNT(*) AS n FROM ist").fetchone())
con.sql("DESCRIBE ist").df()

(19435,)


,column_name,column_type,null,key,default,extra
0,HOSPNUM,BIGINT,YES,None,None,None
1,RDELAY,BIGINT,YES,None,None,None
2,RCONSC,VARCHAR,YES,None,None,None
3,SEX,VARCHAR,YES,None,None,None
4,AGE,BIGINT,YES,None,None,None
...,...,...,...,...,...,...
107,HTI14,BIGINT,YES,None,None,None
108,PE14,BIGINT,YES,None,None,None
109,DVT14,BIGINT,YES,None,None,None
110,TRAN14,BIGINT,YES,None,None,None
